# Masai Interactive connects three analytics sources

## What this shows
How Masai Interactive pulls data from three external analytics sources through one consistent pattern — Google Analytics, Snowflake, and data.world. Every connector exposes the same query → DataFrame shape, and every authentication path is named (service account, config file, OAuth) rather than assumed.

## Why it matters
Web / social analytics work for Masai's clients routinely combines GA events, warehouse rollups, and externally-published public data. A consistent connector interface means the analysis code stays the same across providers; only the credential bootstrap changes.

## Prereqs
- `pip install 'siege-utilities[analytics]'`
- **GA:** service-account JSON (or a 1Password item) — optional
- **Snowflake:** a YAML config file with host / user / private-key — optional
- **data.world:** a `DW_AUTH_TOKEN` env var — optional

## Next
- `analytics/02_ga_end_to_end.ipynb` — full GA pipeline landing in a branded PDF digest for a Masai client.
- `reports/02_slides_pptx_and_google.ipynb` — the delivery surface Masai hands to that client's comms team.


## 1. The connector pattern — three providers, one call shape

Each connector takes auth (any of service account / config file / env var), exposes a `.query()` or `.retrieve()`-style method, and returns a pandas DataFrame. The cell below walks the constructor shape without actually calling out — so the call shape always runs, credentials or not.

In [1]:
from siege_utilities.analytics.google_analytics import (
    GoogleAnalyticsConnector, create_ga_connector_with_service_account,
)
from siege_utilities.analytics.snowflake_connector import get_snowflake_connector
from siege_utilities.analytics.datadotworld_connector import get_datadotworld_connector

print('GA call shape         : create_ga_connector_with_service_account(service_account_data)')
print('Snowflake call shape  : get_snowflake_connector(config_file="snowflake.yaml")')
print('data.world call shape : get_datadotworld_connector(config_file="dw.yaml")')


GA call shape         : create_ga_connector_with_service_account(service_account_data)
Snowflake call shape  : get_snowflake_connector(config_file="snowflake.yaml")
data.world call shape : get_datadotworld_connector(config_file="dw.yaml")


## 2. Google Analytics — Masai pulls last 7 days for a client

If a service-account JSON is available (via `GOOGLE_APPLICATION_CREDENTIALS` or 1Password), the next cell runs a live query. Otherwise it prints the exact call shape you'd use when credentials land.

In [2]:
import os
import json as _json
from pathlib import Path

creds_path = os.environ.get('GOOGLE_APPLICATION_CREDENTIALS')
if creds_path and Path(creds_path).exists():
    svc = _json.loads(Path(creds_path).read_text())
    ga = create_ga_connector_with_service_account(svc)
    print('GA connector initialized with service account: OK')
    # Example call shape — Masai's client property id would go here.
    # df = ga.retrieve_analytics_data(
    #     property_id='properties/123456789',
    #     start_date='7daysAgo', end_date='today',
    #     metrics=['sessions', 'screenPageViews'],
    #     dimensions=['date'],
    # )
else:
    print('GOOGLE_APPLICATION_CREDENTIALS not set — showing call shape only:')
    print("    ga = create_ga_connector_with_service_account(json.loads(open(creds_path).read()))")
    print("    df = ga.retrieve_analytics_data(property_id=..., start_date='7daysAgo', end_date='today', metrics=[...], dimensions=[...])")


GOOGLE_APPLICATION_CREDENTIALS not set — showing call shape only:
    ga = create_ga_connector_with_service_account(json.loads(open(creds_path).read()))
    df = ga.retrieve_analytics_data(property_id=..., start_date='7daysAgo', end_date='today', metrics=[...], dimensions=[...])


## 3. Snowflake — warehouse rollup for a Masai client

`get_snowflake_connector` reads a YAML config describing the account, warehouse, role, and credentials (private key or password). The connector exposes `.execute_query(sql)` returning a DataFrame. Config-driven so the same code runs across clients by pointing at different YAMLs.

In [3]:
cfg = os.environ.get('SNOWFLAKE_CONFIG_FILE')
if cfg and Path(cfg).exists():
    sf = get_snowflake_connector(config_file=cfg)
    print('Snowflake connector initialized: OK')
    # df = sf.execute_query("SELECT date, sessions FROM analytics.daily_traffic WHERE date >= DATEADD(day, -7, CURRENT_DATE)")
else:
    print('SNOWFLAKE_CONFIG_FILE not set — call shape:')
    print("    sf = get_snowflake_connector(config_file='snowflake.yaml')")
    print("    df = sf.execute_query('SELECT ... FROM ... WHERE ...')")


SNOWFLAKE_CONFIG_FILE not set — call shape:
    sf = get_snowflake_connector(config_file='snowflake.yaml')
    df = sf.execute_query('SELECT ... FROM ... WHERE ...')


## 4. data.world — public benchmarks

Masai often benchmarks client site metrics against publicly-published datasets. `get_datadotworld_connector` authenticates via `DW_AUTH_TOKEN` (env var) or a config file, and exposes `.query_sql(dataset_id, sql)` and `.load_dataset(dataset_id)`.

In [4]:
token = os.environ.get('DW_AUTH_TOKEN')
if token:
    dw = get_datadotworld_connector()
    print('data.world connector initialized: OK')
    # df = dw.load_dataset('org/dataset-name')
else:
    print('DW_AUTH_TOKEN not set — call shape:')
    print("    dw = get_datadotworld_connector()")
    print("    df = dw.load_dataset('org/some-public-dataset')")


DW_AUTH_TOKEN not set — call shape:
    dw = get_datadotworld_connector()
    df = dw.load_dataset('org/some-public-dataset')


## 5. Common post-processing — shape-agnostic

All three connectors return pandas DataFrames with arbitrary columns. The analysis code after that is provider-agnostic: normalize date columns, compute week-over-week deltas, apply the Masai client's date range. We demonstrate on a tiny synthetic daily-sessions frame so the pattern always runs.

In [5]:
import pandas as pd

daily = pd.DataFrame({
    'date':     pd.date_range('2026-04-01', periods=14, freq='D'),
    'sessions': [120, 135, 128, 140, 162, 180, 175,
                 155, 170, 168, 190, 210, 230, 225],
})
daily['week']    = daily['date'].dt.isocalendar().week
weekly_totals    = daily.groupby('week')['sessions'].sum()
wow_change_pct   = weekly_totals.pct_change() * 100

summary = pd.DataFrame({
    'weekly_sessions': weekly_totals,
    'WoW change %': wow_change_pct.round(1),
})
summary


,weekly_sessions,WoW change %
week,,
14,685,NaN
15,1248,82.2
16,455,-63.5


## Related

- **Source**: `siege_utilities/analytics/` — `google_analytics.py`, `snowflake_connector.py`, `datadotworld_connector.py`, `facebook_business.py`.
- **Tests**: `tests/test_analytics_*.py`, `tests/test_ga_1password_wrapper_deprecation.py`.
- **Next**: `analytics/02_ga_end_to_end.ipynb` wraps the GA connector into a branded PDF digest Masai ships to a client weekly.
